In [1]:
import pandas as pd

# Load cleaned data
df = pd.read_excel(
    "experiment_analysis.xlsx",
    sheet_name="cleaned_data"
)

# Split groups
control = df[df['experiment_group'] == 'Control']
treatment = df[df['experiment_group'] == 'Treatment']

# Function to calculate metrics
def calculate_metrics(group):
    converted_users = group[group['converted_to_paid'] == 1]

    return {
        "User count": len(group),
        "Landing page visit rate": group['visited_landing_page'].mean(),
        "Trial start rate": group['started_trial'].mean(),
        "Onboarding completion rate": group['completed_onboarding'].mean(),
        "Paid conversion rate": group['converted_to_paid'].mean(),
        "Average revenue per user": group['revenue_30d'].mean(),
        "Average revenue per converted user":
            converted_users['revenue_30d'].mean() if len(converted_users) > 0 else 0,
        "Refund rate": group['refund_requested'].mean(),
        "Support ticket rate": (group['support_tickets_30d'] > 0).mean(),
        "Average engagement score": group['engagement_score'].mean(),
        "Average days to convert": converted_users['days_to_convert'].mean()
    }

# Overall Summary
overall_summary = pd.DataFrame({
    "Metric": calculate_metrics(control).keys(),
    "Control": calculate_metrics(control).values(),
    "Treatment": calculate_metrics(treatment).values()
})

# ---------------- Segment Analysis ----------------

# Region
region_analysis = df.groupby(
    ['region', 'experiment_group']
).agg(
    paid_conversion_rate=('converted_to_paid', 'mean'),
    avg_engagement_score=('engagement_score', 'mean')
).reset_index()

# Device Type
device_analysis = df.groupby(
    ['device_type', 'experiment_group']
).agg(
    paid_conversion_rate=('converted_to_paid', 'mean'),
    trial_start_rate=('started_trial', 'mean')
).reset_index()

# Traffic Source
traffic_analysis = df.groupby(
    ['traffic_source', 'experiment_group']
).agg(
    paid_conversion_rate=('converted_to_paid', 'mean'),
    arpu=('revenue_30d', 'mean')
).reset_index()

# Combine segment analyses
segment_analysis = pd.concat(
    [
        region_analysis.assign(segment='Region'),
        device_analysis.assign(segment='Device Type'),
        traffic_analysis.assign(segment='Traffic Source')
    ],
    ignore_index=True
)

# Save Excel file
with pd.ExcelWriter("experiment_summary.xlsx") as writer:
    overall_summary.to_excel(writer,
                             sheet_name="overall_summary",
                             index=False)

    segment_analysis.to_excel(writer,
                              sheet_name="segment_analysis",
                              index=False)

print("experiment_summary.xlsx created successfully.")

# Display summary
overall_summary

experiment_summary.xlsx created successfully.


,Metric,Control,Treatment
0,User count,690.000000,710.000000
1,Landing page visit rate,0.636232,0.723944
2,Trial start rate,0.250725,0.290141
3,Onboarding completion rate,0.156522,0.211268
4,Paid conversion rate,0.031884,0.070423
5,Average revenue per user,51.974319,54.254521
6,Average revenue per converted user,1630.103636,770.414200
7,Refund rate,0.000000,0.004225
8,Support ticket rate,0.147826,0.247887
9,Average engagement score,57.054493,62.905775
